In [1]:
# -*- coding: utf-8 -*-
"""
Created on Mon Jun  8 14:52:00 2020

This is a test file to test the Hypoid class and its methods.

The Hypoid class is defined in the hypoid.py file.

"""
from hypoid import *
from general_utils import dictprint, dataclass_print
import numpy as np
import screwCalculus as sc
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2
%matplotlib auto

np.set_printoptions(precision=4)

Using matplotlib backend: <object object at 0x000001EA5FC69700>


In [2]:

SystemData = {
    'HAND': "Left",
    'taper' : "Standard",
    'hypoidOffset' : 0,
    'gearGenType' : "generated"
}

met = 4.169

coneData = {
    'SIGMA' : 95,
    'a' : SystemData['hypoidOffset'],
    'z1' : 19,
    'z2' : 62,
    'b2' : 36.83,
    'betam1' : 35,
    'rc0' : 95.25,
    'gearBaseThick' : 15,
    'pinBaseThick' : 8,
}

coneData['de2'] = coneData['z2'] * met
coneData['u'] = coneData['z2']/coneData['z1']

toothData = {
    'alphaD' : 20,
    'alphaC' : 20,
    'falphalim' : 1,
    'khap' : 1,
    'khfp' : 1.25,
    'xhm1' : 0.05,
    'jen' : 0.1,
    'xsmn' : 0.0649,
    'thetaa2' : None,
    'thetaf2' : None
}

H = Hypoid().from_macro_geometry(SystemData, toothData, coneData)
H.plot('pinion', 'both', whole_gear=True)
H.plot('gear', 'both', whole_gear=True)

In [3]:
# testing design data saving and loading functionality
filename = 'basic_data.json'
H.save_design_data_json(filename, 'basic')

# load the design data from json file and instantiate a new Hypoid object
H2 = Hypoid().from_file(filename)
H2.plot('pinion', 'both')


In [4]:
print(H.zRbounds.pinion.concave)
print(H.zRwithRoot.pinion)

print(H.designData.pinion_machine_settings.concave)

[[ 91.7161  26.701 ]
 [127.1969  36.643 ]
 [125.159   43.1399]
 [ 90.387   30.9383]]
[[ 92.011   25.7605]
 [127.4942  35.695 ]
 [125.159   43.1399]
 [ 90.387   30.9383]]
RADIALSETTING: 97.9594
TILTANGLE: 0.0000
SWIVELANGLE: 0.0000
BLANKOFFSET: 0.0000
SLIDINGBASE: -0.0000
CRADLEANGLE: 52.7967
MACHCTRBACK: -0.0013
ROOTANGLE: 15.6410
RATIOROLL: 3.3397
C2: 0.0000
D6: 0.0000
E24: 0.0000
F120: 0.0000
G720: 0.0000
H5040: 0.0000
H1: 0.0000
H2: 0.0000
H3: 0.0000
H4: 0.0000
H5: 0.0000
H6: 0.0000
V1: 0.0000
V2: 0.0000
V3: 0.0000
V4: 0.0000
V5: 0.0000
V6: 0.0000
R1: 0.0000
R2: 0.0000
R3: 0.0000
R4: 0.0000
R5: 0.0000
R6: 0.0000
TLT1: 0.0000
TLT2: 0.0000
TLT3: 0.0000
TLT4: 0.0000
TLT5: 0.0000
TLT6: 0.0000
SW1: 0.0000
SW2: 0.0000
SW3: 0.0000
SW4: 0.0000
SW5: 0.0000
SW6: 0.0000



In [27]:
np.set_printoptions(precision=4)
# if you dont remember the indexing just call
# dictprint(H.get_machine_settings_names())
x_index = [0,3,4,5,7,15,24,33,
           72,74,75]
# x_index = [0,1,2,3,4,5,7,15,
#            72,73,74,77]

x_index.sort()
lb, ub = H.compute_identification_bounds('pinion', 'concave', x_index)


In [28]:
solver, settings = H.buildIdentificationProblem('pinion', 'concave', x_index, lb, ub, zR=None, problem_type= 'ease-off', bound_points_tol=1)


In [29]:
# import matlab.engine
# eng = matlab.engine.start_matlab()

# compute the target points
v5DoF = np.array([0,0,50,25,0])/1000 # PA, SA, PC, LC, TW
# v5DoF = np.array([1,0,2,4,0]) # PA, SA, PC, LC, TW
E_fun = ease_off_5DoF(v5DoF)

num_profile, num_face = 11, 22
U, V = np.meshgrid(np.linspace(-1,1,num_face), np.linspace(-1, 1, num_profile))
E = E_fun(U, V)

Z, R = H.compute_zr_grid('pinion', 'concave', 11, 22)

base_points = H.identificationProblemEaseOff.pinion.concave['base_points'].squeeze()
base_normals = H.identificationProblemEaseOff.pinion.concave['base_normals'].squeeze()
target_points = base_points + (E.flatten(order = 'F')) * base_normals

EO = np.sum((target_points - base_points) * base_normals, axis = 0)
plot_ease_off(EO.reshape(Z.shape, order = 'F'), Z, R, aspect_ratio=[1,1,0.010], labels=['z (mm)', 'R (mm)', 'E ($\mu$m)'])
root_points = H.identificationProblemEaseOff.pinion.concave['root_constraint']['points']

target_points = np.concatenate((target_points[0:3,:], root_points[0:3,:]), 1)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(target_points[0,:], target_points[1,:], target_points[2,:])
ax.scatter(base_points[0,:], base_points[1,:], base_points[2,:])
# set axis equal
ax.axis('equal')

# set axis labels
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()



<>:20: SyntaxWarning: invalid escape sequence '\m'
<>:20: SyntaxWarning: invalid escape sequence '\m'
C:\Users\egrab\AppData\Local\Temp\ipykernel_72216\2222587293.py:20: SyntaxWarning: invalid escape sequence '\m'
  plot_ease_off(EO.reshape(Z.shape, order = 'F'), Z, R, aspect_ratio=[1,1,0.010], labels=['z (mm)', 'R (mm)', 'E ($\mu$m)'])


In [30]:
new_settings = evaluate_identification_problem(solver, settings, target_points)


This is Ipopt version 3.14.11, running with linear solver ma57.

Number of nonzeros in equality constraint Jacobian...:    41394
Number of nonzeros in inequality constraint Jacobian.:      144
Number of nonzeros in Lagrangian Hessian.............:    16540

Total number of variables............................:     5671
                     variables with only lower bounds:        0
                variables with lower and upper bounds:     5671
                     variables with only upper bounds:        0
Total number of equality constraints.................:     5628
Total number of inequality constraints...............:       48
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:       48
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.22e-01 0.00e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00   0
   

In [31]:
identified_data = H.identificationProblemEaseOff.designData.update_settings('pinion', 'concave', x_index, new_settings, return_copy=True)

print(identified_data.pinion_cutter_data.concave)
print('\n\n')
print(H.designData.pinion_cutter_data.concave)

POINTRADIUS: 98.6910
EDGERADIUS: 0.2084
TYPE: CURVED
RHO: 233.8404
BLADEANGLE: 19.8823
topremTYPE: NONE
topremDEPTH: 0.0000
topremRADIUS: 0.0000
topremANGLE: 0.0000
flankremTYPE: NONE
flankremDEPTH: 0.0000
flankremRADIUS: 0.0000
flankremANGLE: 0.0000




POINTRADIUS: 95.9517
EDGERADIUS: 0.2084
TYPE: STRAIGHT
RHO: None
BLADEANGLE: 20.0000
topremTYPE: NONE
topremDEPTH: 0.0000
topremRADIUS: 0.0000
topremANGLE: 0.0000
flankremTYPE: NONE
flankremDEPTH: 0.0000
flankremRADIUS: 0.0000
flankremANGLE: 0.0000



In [ ]:
n_face = 10
n_flank = 20
z,R = H.compute_zr_grid('pinion', 'concave', n_face, n_flank, active_flank=True)
zR = np.vstack((z.flatten(order='F'), R.flatten(order = 'F'))).T
# %matplotlib qt
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(zR[:,0], zR[:,1], c = 'blue')
ax.axis('equal')
plt.show()




In [ ]:
H.surfTriplets.gear.concave[2,:]

In [ ]:
print(zR.shape)
print(H.designData.system_data.hypoid_offset)
p, n, triplets_conj, zRconj, psi_P, angular_ease_off, v_pg_p, omega, psi_G =  H.compute_conjugate_points_to_gear('concave', zR, [0,0,0,0], 0)

In [ ]:
X = p[0,:].reshape(n_face, n_flank, order='F').T
Y = p[1,:].reshape(n_face, n_flank, order='F').T
Z = p[2,:].reshape(n_face, n_flank, order='F').T

import easy_plot as ep
F = ep.Figure()
S = ep.surface(F,X, Y, Z)
F.show()

In [154]:

H.plot_zr_bounds('pinion', 'concave')

In [ ]:

z, R = H.compute_zr_grid('pinion', 'concave', 20, 30)

fig = plt.figure()
ax = fig.add_subplot(111)
ax.pcolormesh(z, R, np.zeros_like(z), edgecolors='k', linewidth=0.5)
plt.axis('equal')
H.plot_zr_bounds('pinion', 'concave')

H.zRbounds.pinion.concave[0,:]



In [ ]:
points, _, _ = H.samplezR([], [], 'gear', 'concave')
points_cvx, _, _ = H.samplezR([], [], 'gear', 'convex')
print(points.shape)

points = np.squeeze(points)
print(points.shape)
import matplotlib.pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points[0,:], points[1,:], points[2,:])
ax.scatter(points_cvx[0,:], points_cvx[1,:], points_cvx[2,:])
# set axis equal
ax.axis('equal')

# set axis labels
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()



In [ ]:
from hypoid.main.utils import machine_settings_index
print(H.get_settings_index(machine_settings_names=['RADIALSETTING', 'SPHERICALRADIUS']))

dd = H.get_machine_settings_names()

dictprint(machine_settings_index(completing=True))


In [ ]:
# identification problem set-up

from hypoid.main.identification import identification_bounds
settings_index = [0, 3, 4, 5, 15]

machine_UB, machine_LB, cutter_UB, cutter_LB = identification_bounds(H.designData, 'pinion', 'concave')

ub, lb = bounds_from_settings(settings_index)

In [ ]:
from PyQt5 import QtWidgets, QtCore
import sys

app = QtWidgets.QApplication([])

dark_stylesheet = """
QWidget {
    background-color: #2b2b2b;
    color: #f0f0f0;
}

QPushButton {
    background-color: #3c3c3c;
    border: 1px solid #555555;
    padding: 5px;
}

QPushButton:hover {
    background-color: #444444;
}
"""

app.setStyleSheet(dark_stylesheet)

class SettingsSelector(QtWidgets.QDialog):
    def __init__(self, settings_list):
        super().__init__()
        self.setWindowTitle("Select Optimization Settings")

        self.settings_list = settings_list
        self.selected_settings = []
        self.bounds = {}

        self.layout = QtWidgets.QVBoxLayout()
        self.table = QtWidgets.QTableWidget(len(settings_list), 4)
        self.table.setHorizontalHeaderLabels(["Select", "Current value", "Min Bound", "Max Bound"])

        # Set background color for table cells
        self.table.setStyleSheet("""
        QTableWidget {
            background-color: #2b2b2b;
            gridline-color: #555555;
            color: #f0f0f0;
        }

        QHeaderView::section {
            background-color: #3c3c3c;
            color: #f0f0f0;
        }
        """)
        for i, s in enumerate(settings_list):
            # Checkbox
            chk_item = QtWidgets.QTableWidgetItem()
            chk_item.setFlags(QtCore.Qt.ItemFlag.ItemIsUserCheckable | QtCore.Qt.ItemFlag.ItemIsEnabled)
            chk_item.setCheckState(QtCore.Qt.CheckState.Unchecked)
            self.table.setItem(i, 0, chk_item)
            
            # Current value
            current_item = QtWidgets.QTableWidgetItem("0")
            self.table.setItem(i, 1, current_item)

            # Min Bound
            min_item = QtWidgets.QTableWidgetItem("0")
            self.table.setItem(i, 2, min_item)
            
            # Max Bound
            max_item = QtWidgets.QTableWidgetItem("10")
            self.table.setItem(i, 3, max_item)

        self.layout.addWidget(self.table)

        # Buttons
        btn_layout = QtWidgets.QHBoxLayout()
        ok_btn = QtWidgets.QPushButton("OK")
        ok_btn.clicked.connect(self.ok_clicked)
        cancel_btn = QtWidgets.QPushButton("Cancel")
        cancel_btn.clicked.connect(self.reject)
        btn_layout.addWidget(ok_btn)
        btn_layout.addWidget(cancel_btn)
        self.layout.addLayout(btn_layout)

        self.setLayout(self.layout)

    def ok_clicked(self):
        for i, s in enumerate(self.settings_list):
            item = self.table.item(i, 0)
            if item.checkState() == QtCore.Qt.CheckState.Checked:
                self.selected_settings.append(s)
                min_val = float(self.table.item(i, 1).text())
                max_val = float(self.table.item(i, 2).text())
                self.bounds[s] = (min_val, max_val)
        self.accept()

def select_settings(settings_list):
    app = QtWidgets.QApplication(sys.argv)
    dialog = SettingsSelector(settings_list)
    if dialog.exec() == QtWidgets.QDialog.DialogCode.Accepted:
        return dialog.selected_settings, dialog.bounds
    else:
        return [], {}

# Example usage
if __name__ == "__main__":
    settings = ["feed_rate", "spindle_speed", "tool_diameter", "cut_depth"]
    sel, bnds = select_settings(settings)
    print(sel)
    print(bnds)
